# Waymax batch dataloading and preprocessing timing

이 노트북은 `waymax` dataloader로 배치를 읽는 시간과 `data/preprocess.py`의 `preprocess_simulator_state` 전처리 시간을 분리해서 측정합니다.

실행 전에 `tfrecord_path`만 현재 환경의 Waymo TFRecord shard 경로로 바꾸면 됩니다.

In [7]:
from __future__ import annotations

import dataclasses
import json
from pathlib import Path
from time import perf_counter

import jax
import numpy as np

from data.preprocess import preprocess_simulator_state
from data.types import PreprocessConfig
from waymax import config as waymax_config
from waymax import dataloader


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# Set this to a Waymo/WAYMAX TFRecord shard. If you point it at a directory,
# the first matching shard is used.
tfrecord_path = "/zfsauton/scratch/eshau/womd/tf_example/training/training_tfexample.tfrecord-00000-of-01000"

batch_size = 4
max_num_objects = 64
shuffle_seed = 0
num_timed_batches = 20
warmup_batches = 2
preprocess_cfg = PreprocessConfig()
preprocess_seed = 0


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def resolve_tfrecord_path(raw_path: str) -> str:
    path = Path(raw_path)
    if path.is_file():
        return str(path)

    candidates = sorted(path.glob("training_tfexample.tfrecord-*-of-*"))
    if not candidates:
        raise FileNotFoundError(
            f"No tfrecord shard found at {raw_path}. Set tfrecord_path to a shard file or a directory containing shards."
        )
    return str(candidates[0])


def block_until_ready_tree(tree) -> None:
    for leaf in jax.tree_util.tree_leaves(tree):
        if hasattr(leaf, "block_until_ready"):
            leaf.block_until_ready()


def summarize_ms(values_s: list[float]) -> dict[str, float | int]:
    arr = np.asarray(values_s, dtype=np.float64) * 1000.0
    if arr.size == 0:
        return {"count": 0}
    return {
        "count": int(arr.size),
        "mean_ms": float(arr.mean()),
        "std_ms": float(arr.std(ddof=0)),
        "min_ms": float(arr.min()),
        "p50_ms": float(np.percentile(arr, 50)),
        "p90_ms": float(np.percentile(arr, 90)),
        "p95_ms": float(np.percentile(arr, 95)),
        "max_ms": float(arr.max()),
        "throughput_batches_per_s": float(1.0 / max(arr.mean() / 1000.0, 1e-12)),
    }


print("Helpers are ready. Run the next cell after setting `tfrecord_path`.")

Helpers are ready. Run the next cell after setting `tfrecord_path`.


In [9]:
# Build the Waymax dataloader and benchmark loading + preprocessing.
resolved_tfrecord_path = resolve_tfrecord_path(tfrecord_path)

ds_cfg = dataclasses.replace(
    waymax_config.WOD_1_3_1_TRAINING,
    path=resolved_tfrecord_path,
    max_num_objects=int(max_num_objects),
    batch_dims=(int(batch_size),),
    shuffle_seed=int(shuffle_seed),
)

state_iter = dataloader.simulator_state_generator(ds_cfg)
rng = jax.random.PRNGKey(int(preprocess_seed))

print("resolved_tfrecord_path:", resolved_tfrecord_path)
print("batch_size:", batch_size)
print("max_num_objects:", max_num_objects)
print("num_timed_batches:", num_timed_batches)
print("warmup_batches:", warmup_batches)

load_times_s: list[float] = []
preprocess_times_s: list[float] = []
total_times_s: list[float] = []
observed_batch_shape = None
observed_feature_shapes = None
observed_aux_shapes = None

for batch_index in range(int(warmup_batches) + int(num_timed_batches)):
    loop_start = perf_counter()

    load_start = perf_counter()
    sim_state = next(state_iter)
    load_end = perf_counter()

    rng, key = jax.random.split(rng)
    preprocess_start = perf_counter()
    pre_batch, _ = preprocess_simulator_state(sim_state, key, preprocess_cfg)
    block_until_ready_tree((pre_batch.features, pre_batch.aux))
    preprocess_end = perf_counter()

    if observed_batch_shape is None:
        observed_batch_shape = tuple(sim_state.log_trajectory.x.shape)
        observed_feature_shapes = {key: tuple(value.shape) for key, value in pre_batch.features.items()}
        observed_aux_shapes = {key: tuple(value.shape) for key, value in pre_batch.aux.items()}

    if batch_index >= int(warmup_batches):
        load_times_s.append(load_end - load_start)
        preprocess_times_s.append(preprocess_end - preprocess_start)
        total_times_s.append(preprocess_end - loop_start)

summary = {
    "config": {
        "tfrecord_path": resolved_tfrecord_path,
        "batch_size": int(batch_size),
        "max_num_objects": int(max_num_objects),
        "num_timed_batches": int(num_timed_batches),
        "warmup_batches": int(warmup_batches),
    },
    "observed_batch_shape": observed_batch_shape,
    "observed_feature_shapes": observed_feature_shapes,
    "observed_aux_shapes": observed_aux_shapes,
    "load_time": summarize_ms(load_times_s),
    "preprocess_time": summarize_ms(preprocess_times_s),
    "total_time": summarize_ms(total_times_s),
}

print(json.dumps(summary, indent=2, sort_keys=True))

resolved_tfrecord_path: /zfsauton/scratch/eshau/womd/tf_example/training/training_tfexample.tfrecord-00000-of-01000
batch_size: 4
max_num_objects: 64
num_timed_batches: 20
warmup_batches: 2
{
  "config": {
    "batch_size": 4,
    "max_num_objects": 64,
    "num_timed_batches": 20,
    "tfrecord_path": "/zfsauton/scratch/eshau/womd/tf_example/training/training_tfexample.tfrecord-00000-of-01000",
    "warmup_batches": 2
  },
  "load_time": {
    "count": 20,
    "max_ms": 3.5229213535785675,
    "mean_ms": 3.165513672865927,
    "min_ms": 2.6106461882591248,
    "p50_ms": 3.1545280944556,
    "p90_ms": 3.506117593497038,
    "p95_ms": 3.5121383843943477,
    "std_ms": 0.24166198662866412,
    "throughput_batches_per_s": 315.90449555526345
  },
  "observed_aux_shapes": {
    "anchor_step": [
      4
    ],
    "anchor_timestamp_micros": [
      4
    ],
    "anchor_world_state": [
      4,
      5
    ],
    "anchor_yaw": [
      4
    ],
    "ego_index": [
      4
    ],
    "model_t_se